<a href="https://colab.research.google.com/github/it0770e/xai-ids/blob/main/03_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# CELL 1: Mount Drive and setup
# ============================================
from google.colab import drive
drive.mount('/content/drive')

# Clone repo for any updates
!git clone https://github.com/i10770e/xai-ids.git
%cd xai-ids

import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

print("✅ Setup complete!")

Mounted at /content/drive
Cloning into 'xai-ids'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'xai-ids'
/content
✅ Setup complete!


In [ ]:
# ============================================
# CELL 2: Load NSL-KDD data from Drive
# ============================================

# Load the data we saved in Notebook 2
train_path = '/content/drive/MyDrive/xai-ids/data/raw/nsl_kdd_train.pkl'
test_path = '/content/drive/MyDrive/xai-ids/data/raw/nsl_kdd_test.pkl'

df_train = pd.read_pickle(train_path)
df_test = pd.read_pickle(test_path)

print(f"✅ Training data loaded: {df_train.shape}")
print(f"✅ Test data loaded: {df_test.shape}")
print(f"\nColumns in dataset: {df_train.columns.tolist()}")

✅ Training data loaded: (125973, 45)
✅ Test data loaded: (22544, 45)

Columns in dataset: ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty', 'binary_label', 'attack_category']


In [ ]:
# ============================================
# CELL 3: Separate features from labels
# ============================================

# For binary classification (our main task)
y_train = df_train['binary_label'].values
y_test = df_test['binary_label'].values

# Drop label columns to get features
X_train_raw = df_train.drop(columns=['label', 'binary_label', 'attack_category', 'difficulty'])
X_test_raw = df_test.drop(columns=['label', 'binary_label', 'attack_category', 'difficulty'])

print(f"✅ Features shape (train): {X_train_raw.shape}")
print(f"✅ Features shape (test): {X_test_raw.shape}")
print(f"\nFeature columns: {X_train_raw.columns.tolist()}")

✅ Features shape (train): (125973, 41)
✅ Features shape (test): (22544, 41)

Feature columns: ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate']


In [ ]:
# ============================================
# CELL 4: Encode categorical features
# ============================================

# Identify categorical columns
categorical_cols = X_train_raw.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_train_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"📊 Categorical columns: {categorical_cols}")
print(f"📊 Numerical columns: {len(numerical_cols)} columns")

# Encode categorical features
label_encoders = {}
X_train_encoded = X_train_raw.copy()
X_test_encoded = X_test_raw.copy()

for col in categorical_cols:
    le = LabelEncoder()
    # Fit on training data only
    X_train_encoded[col] = le.fit_transform(X_train_raw[col])
    # Transform test data
    X_test_encoded[col] = le.transform(X_test_raw[col])
    label_encoders[col] = le

print("✅ Categorical features encoded")

📊 Categorical columns: ['protocol_type', 'service', 'flag']
📊 Numerical columns: 38 columns
✅ Categorical features encoded


In [ ]:
# ============================================
# CELL 5: Scale features (normalize)
# ============================================

scaler = StandardScaler()

# Fit on training data, transform both train and test
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

print(f"✅ Features scaled")
print(f"   Train shape: {X_train_scaled.shape}")
print(f"   Test shape: {X_test_scaled.shape}")
print(f"\n   Mean after scaling (should be ~0): {X_train_scaled.mean(axis=0)[:5]}")
print(f"   Std after scaling (should be ~1): {X_train_scaled.std(axis=0)[:5]}")

✅ Features scaled
   Train shape: (125973, 41)
   Test shape: (22544, 41)

   Mean after scaling (should be ~0): [ 2.54947740e-17 -6.19601974e-17  6.40753612e-17  8.38732941e-17
 -4.51234938e-19]
   Std after scaling (should be ~1): [1. 1. 1. 1. 1.]


In [ ]:
# ============================================
# CELL 6: Split training into train + validation
# ============================================

# Split training data (70% train, 15% val, 15% test)
# We already have test set, so split training into train/val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_scaled, y_train,
    test_size=0.18,  # 18% of 85% ≈ 15% of total
    random_state=42,
    stratify=y_train
)

print(f"✅ Final splits:")
print(f"   Training set: {X_train.shape}")
print(f"   Validation set: {X_val.shape}")
print(f"   Test set: {X_test_scaled.shape}")

✅ Final splits:
   Training set: (103297, 41)
   Validation set: (22676, 41)
   Test set: (22544, 41)


In [ ]:
# ============================================
# CELL 7: Apply SMOTE to balance classes
# ============================================

# Check current class distribution
print("📊 Before SMOTE:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"   Class {u}: {c} samples ({c/len(y_train):.1%})")

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("\n📊 After SMOTE:")
unique, counts = np.unique(y_train_balanced, return_counts=True)
for u, c in zip(unique, counts):
    print(f"   Class {u}: {c} samples ({c/len(y_train_balanced):.1%})")

📊 Before SMOTE:
   Class 0: 55221 samples (53.5%)
   Class 1: 48076 samples (46.5%)

📊 After SMOTE:
   Class 0: 55221 samples (50.0%)
   Class 1: 55221 samples (50.0%)


In [ ]:
# ============================================
# CELL 8: Save processed data and preprocessors
# ============================================

# Save the processed data
processed_data = {
    'X_train': X_train_balanced,
    'X_val': X_val,
    'X_test': X_test_scaled,
    'y_train': y_train_balanced,
    'y_val': y_val,
    'y_test': y_test
}

save_path = '/content/drive/MyDrive/xai-ids/data/processed/nsl_kdd_processed.pkl'
joblib.dump(processed_data, save_path)
print(f"✅ Processed data saved to: {save_path}")

# Save the preprocessors for later use
preprocessors = {
    'label_encoders': label_encoders,
    'scaler': scaler,
    'categorical_cols': categorical_cols,
    'numerical_cols': numerical_cols
}

preprocessor_path = '/content/drive/MyDrive/xai-ids/results/models/preprocessors.pkl'
joblib.dump(preprocessors, preprocessor_path)
print(f"✅ Preprocessors saved to: {preprocessor_path}")

✅ Processed data saved to: /content/drive/MyDrive/xai-ids/data/processed/nsl_kdd_processed.pkl
✅ Preprocessors saved to: /content/drive/MyDrive/xai-ids/results/models/preprocessors.pkl


In [ ]:
# ============================================
# CELL 9: Verify everything is ready
# ============================================

print("📊 FINAL CHECK:")
print(f"   Training data: {X_train_balanced.shape}")
print(f"   Training labels: {y_train_balanced.shape}")
print(f"   Validation data: {X_val.shape}")
print(f"   Test data: {X_test_scaled.shape}")
print(f"\n   Class distribution in training: {np.bincount(y_train_balanced)}")
print(f"   Class distribution in validation: {np.bincount(y_val)}")
print(f"   Class distribution in test: {np.bincount(y_test)}")

print("\n🎉 PREPROCESSING COMPLETE! Ready for modeling!")

📊 FINAL CHECK:
   Training data: (110442, 41)
   Training labels: (110442,)
   Validation data: (22676, 41)
   Test data: (22544, 41)

   Class distribution in training: [55221 55221]
   Class distribution in validation: [12122 10554]
   Class distribution in test: [ 9711 12833]

🎉 PREPROCESSING COMPLETE! Ready for modeling!
